# 05 · Direction 1: richer product input and a stronger backbone

**Question:** how much does the cross-encoder gain from (a) adding brand, color and bullet points to the product text, and (b) replacing MiniLM-L12 with `deberta-v3-base`? Both are measured against B3 (MiniLM, title only, MSE), on validation.

| Run | Model | Product text | Seeds | Compared with |
|---|---|---|---|---|
| `d1_minilm_title_bc_mse` | MiniLM (B3 settings) | `title_bc` = title + brand + color | 42, 43 | B3, same seed |
| `d1_minilm_main_mse` | MiniLM (B3 settings) | `main` = `title_bc` + bullets | 42, 43 | B3, same seed |
| `d1_deberta_title_mse` | DeBERTa-v3-base | `title` | 42 | B3, same seed |
| `d1_deberta_main_mse` | DeBERTa-v3-base | `main` | 42, 43 | MiniLM `main`, same seed |

**Settings.** MiniLM keeps B3's settings exactly (lr 7e-6, 5,000 warm-up steps), so only the input changes. DeBERTa starts with an untrained regression head, so it uses lr 2e-5 with about 10% warm-up (1,200 of ~11,800 steps). Both: MSE on gains, 1 epoch, batch 32, fp16. `max_len` 128 for `title` / `title_bc`, 256 for `main` (section 1 prints the share of truncated pairs with each real tokenizer).

**Test is blind.** Configurations are compared on val only. Test scores are predicted and saved (`scores/<run>_test.parquet`) but not scored here; `10_report` scores test once the final configurations are chosen.

**Kaggle settings:** GPU **T4**, **Internet on** (Hugging Face models).

**Inputs:**
- Dataset `wm-src` (zip of `src/` and `data/results/`, see README). Required. Its `data/results/` holds the B3 scores used for the paired comparisons.
- Add Input → Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.

**How to run.** The runs are split into three batches so that no session is long (T4 hours per batch, training + val/test prediction):

| `BATCH` | Runs | Approx. time |
|---|---|---|
| `'minilm'` | MiniLM `title_bc` and `main`, seeds 42 and 43 | 2.5 h |
| `'deberta_a'` | DeBERTa `title` s42, DeBERTa `main` s42 | 3.5 h |
| `'deberta_b'` | DeBERTa `main` s43 | 2.6 h |

1. `MODE = 'smoke'`, `BATCH = 'all'`: every configuration on 5% of train with seed 42, val only (about 15 minutes). Check the verdict at the end of section 2.
2. `MODE = 'full'`: set `BATCH`, then **Save Version → Save & Run All**; one version per batch. After each version, download `results/` into `data/results/` in the repo.
3. For the comparison of DeBERTa `main` with MiniLM `main`, the MiniLM `main` scores must be in `wm-src` (rebuild the zip after step 2 of batch `minilm`) or attached as this notebook's earlier output.

**Output** (`/kaggle/working/results/`): `<run>.json` (val metrics, config, val curve, `src_version`) and `scores/<run>_{val,test}.parquet`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer

from src.cross_encoder import predict, train_ce
from src.data import load_pairs, product_texts
from src.evaluation import RunLog

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'    # 'smoke': 5% of train, seed 42 only, val only | 'full': all train, the seeds below
BATCH = 'all'     # 'all' (smoke only) | 'minilm' | 'deberta_a' | 'deberta_b'

BASE = {'loss': 'mse', 'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True}
MODELS = {
    'minilm': {**BASE, 'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'lr': 7e-6, 'warmup_steps': 5000},   # = B3
    'deberta': {**BASE, 'model': 'microsoft/deberta-v3-base', 'lr': 2e-5, 'warmup_steps': 1200},             # ~10% warm-up
}
MAX_LEN = {'title': 128, 'title_bc': 128, 'main': 256}
EVAL_BATCH = {('deberta', 'main'): 64}   # pairs per prediction batch (default 512); 512 runs out of memory here

# (model, fields, seeds, reference run to compare with; {seed} is filled in)
PLAN = {
    'minilm': [('minilm', 'title_bc', [42, 43], 'b3_minilm_title_mse_s{seed}'),
               ('minilm', 'main', [42, 43], 'b3_minilm_title_mse_s{seed}')],
    'deberta_a': [('deberta', 'title', [42], 'b3_minilm_title_mse_s{seed}'),
                  ('deberta', 'main', [42], 'd1_minilm_main_mse_s{seed}')],
    'deberta_b': [('deberta', 'main', [43], 'd1_minilm_main_mse_s{seed}')],
}
assert MODE in ('smoke', 'full') and BATCH in PLAN or (MODE == 'smoke' and BATCH == 'all')
JOBS = [j for b in (PLAN if BATCH == 'all' else [BATCH]) for j in PLAN[b]]
if MODE == 'smoke':   # each configuration once, seed 42, no paired comparison (5% of train is not comparable)
    JOBS = list({(m, f): (m, f, [42], None) for m, f, _, _ in JOBS}.values())

SEED = 42
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']   # test is predicted and saved, never scored here
EVAL_EVERY = 100 if MODE == 'smoke' else 4000                   # steps between val checks (monitoring only)
RES = Path('/kaggle/working/results')

assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| BATCH', BATCH)
for m, f, seeds, ref in JOBS:
    print(f'  {m:8s} {f:9s} max_len {MAX_LEN[f]}  seeds {seeds}  vs {ref}')

## 1. Data

`load_pairs` reads `pairs.parquet`; the product text of each preset comes from `products_clean.parquet` (`src/data.py`). The truncation check uses the real tokenizer of each model on 20,000 random pairs.

In [ ]:
pairs = load_pairs('title')
FIELDS = sorted({f for _, f, _, _ in JOBS})
TEXT = {f: product_texts(f) for f in FIELDS}

MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_all = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_all['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train_all = train_all[train_all['query_id'].isin(keep)]
train_all = train_all.reset_index(drop=True)
print(f'train {train_all["query_id"].nunique():,} queries / {len(train_all):,} pairs | '
      f'val {MASK["val"].sum():,} | test {MASK["test"].sum():,} pairs')


def with_text(df, fields):
    return df.assign(text=df['product_id'].map(TEXT[fields]).to_numpy())


smp = pairs.sample(20_000, random_state=SEED)
trunc = []
for m, f in sorted({(m, f) for m, f, _, _ in JOBS}):
    tok = AutoTokenizer.from_pretrained(MODELS[m]['model'])
    n_tok = np.array([len(x) for x in tok(smp['query'].tolist(), smp['product_id'].map(TEXT[f]).tolist())['input_ids']])
    trunc.append({'model': m, 'fields': f, 'max_len': MAX_LEN[f], 'median_tokens': int(np.median(n_tok)),
                  'p90': int(np.percentile(n_tok, 90)), 'truncated': round(float((n_tok > MAX_LEN[f]).mean()), 3)})
pd.DataFrame(trunc)

## 2. Training runs

Each job trains with `train_ce` (`src/cross_encoder.py`) and logs val metrics with `RunLog`; test scores are saved blind. Reference runs (B3, MiniLM `main`) are loaded from the inputs when present, for the paired bootstrap on val nDCG.

In [ ]:
log = RunLog(RES, {s: pairs.loc[MASK[s], ['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
for ref in sorted({ref.format(seed=s) for _, _, seeds, ref in JOBS if ref for s in seeds}):
    if not log.load(ref, splits=['val']):
        print(f'{ref}: not found in the inputs (no paired comparison)')

runs = []
for m, f, seeds, ref in JOBS:
    cfg = {**MODELS[m], 'fields': f, 'max_len': MAX_LEN[f], 'eval_batch_size': EVAL_BATCH.get((m, f), 512)}
    train_df = with_text(train_all, f)
    parts = {s: with_text(pairs[MASK[s]], f) for s in EVAL_SPLITS}
    for seed in seeds:
        name = f"{'smoke_' if MODE == 'smoke' else ''}d1_{m}_{f}_{cfg['loss']}_s{seed}"
        print(f'\n==== {name}')
        model, tokenizer, info = train_ce(cfg, train_df, seed, eval_df=parts['val'], eval_every=EVAL_EVERY,
                                          print_every=100 if MODE == 'smoke' else 1000)
        t0 = time.time()
        torch.cuda.empty_cache()
        scores = {s: predict(model, tokenizer, parts[s], cfg['max_len'], batch_size=cfg['eval_batch_size'],
                             fp16=cfg['fp16']) for s in EVAL_SPLITS}
        log.log(name, scores, {'mode': MODE, **cfg, 'seed': seed, 'subsample': SUBSAMPLE,
                               'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
                               'gpu': torch.cuda.get_device_name(0), 'predict_minutes': round((time.time() - t0) / 60, 1),
                               **info},
                compare_to=ref.format(seed=seed) if ref else None, blind=('test',))
        runs.append(name)
        del model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if MODE == 'smoke':
    ok = True
    for n in runs:
        r = log.records[n]
        first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
        v = r['metrics']['val']['ndcg']
        full_h = r['steps'] / SUBSAMPLE / r['it_per_s'] / 3600
        good = last < first and v > 0.742
        ok &= good
        print(f"{'OK   ' if good else 'CHECK'} {n:34s} loss {first:.4f} -> {last:.4f} | val nDCG {v} | "
              f"full epoch ~{full_h:.2f} h")
    print('Smoke test passed: switch MODE to full.' if ok else
          'Some runs did not learn (loss up or val at random level): send me this output.')

## 3. Results (val)

`paired_vs`: val nDCG difference with the reference run over the same queries; a CI that excludes 0 means a significant difference. With 5% of train (smoke) the numbers only show that training works, not which configuration is better.

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
log.print_paired()

curves = pd.DataFrame([{'run': n, **c} for n in runs for c in log.records[n]['val_curve']])
if len(curves):
    display(curves.pivot(index='step', columns='run', values='val_ndcg'))